# Module 40: Building an Image Classifier — End-to-End Computer Vision

Build a complete image classifier from scratch: data pipeline, augmentation, CNN/ResNet, training with AMP, evaluation, TTA, and Grad-CAM.

**No pretrained weights** — every component is implemented from scratch.

| Input | Output |
|-------|--------|
| `32x32 RGB image of circle` | `circle (0.95)` |
| `32x32 RGB image of star` | `star (0.91)` |
| `32x32 RGB image of triangle` | `triangle (0.88)` |

In [ ]:
import math
import random
import time
from collections import defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split

print(f"PyTorch {torch.__version__}")
torch.manual_seed(42)
random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

## 1. Synthetic Shape Dataset

We generate 10 classes of geometric shapes. This lets us train without downloading external data.

In [ ]:
CLASS_NAMES = [
    "circle", "square", "triangle", "cross", "diamond",
    "star", "ring", "arrow", "pentagon", "hexagon",
]
NUM_CLASSES = len(CLASS_NAMES)


def _draw_circle(img, cx, cy, r, val):
    H, W = img.shape[-2:]
    yy, xx = torch.meshgrid(torch.arange(H), torch.arange(W), indexing="ij")
    mask = ((xx - cx).float().pow(2) + (yy - cy).float().pow(2)).sqrt() <= r
    img[:, mask] = val


def _draw_square(img, cx, cy, r, val):
    H, W = img.shape[-2:]
    img[:, max(0, cy-r):min(H, cy+r), max(0, cx-r):min(W, cx+r)] = val


def _draw_triangle(img, cx, cy, r, val):
    H, W = img.shape[-2:]
    yy, xx = torch.meshgrid(torch.arange(H), torch.arange(W), indexing="ij")
    top_y, base_y = cy - r, cy + r
    height = (yy - top_y).float().clamp(min=0)
    half_width = height * (r / max(1, base_y - top_y))
    mask = (yy >= top_y) & (yy <= base_y) & ((xx - cx).float().abs() <= half_width)
    img[:, mask] = val


def _draw_cross(img, cx, cy, r, val):
    t = max(2, r // 3)
    H, W = img.shape[-2:]
    img[:, max(0,cy-r):min(H,cy+r), max(0,cx-t):min(W,cx+t)] = val
    img[:, max(0,cy-t):min(H,cy+t), max(0,cx-r):min(W,cx+r)] = val


def _draw_diamond(img, cx, cy, r, val):
    H, W = img.shape[-2:]
    yy, xx = torch.meshgrid(torch.arange(H), torch.arange(W), indexing="ij")
    mask = ((xx - cx).float().abs() + (yy - cy).float().abs()) <= r
    img[:, mask] = val


def _draw_star(img, cx, cy, r, val):
    _draw_cross(img, cx, cy, r, val)
    _draw_diamond(img, cx, cy, r, val)


def _draw_ring(img, cx, cy, r, val):
    H, W = img.shape[-2:]
    yy, xx = torch.meshgrid(torch.arange(H), torch.arange(W), indexing="ij")
    dist = ((xx - cx).float().pow(2) + (yy - cy).float().pow(2)).sqrt()
    inner = max(1, r - max(2, r // 3))
    mask = (dist >= inner) & (dist <= r)
    img[:, mask] = val


def _draw_arrow(img, cx, cy, r, val):
    t = max(2, r // 4)
    H, W = img.shape[-2:]
    img[:, max(0,cy-t):min(H,cy+t), max(0,cx-r):min(W,cx+r)] = val
    _draw_triangle(img, cx + r - r // 3, cy, r // 2, val)


def _draw_pentagon(img, cx, cy, r, val):
    _draw_circle(img, cx, cy, r, val)
    _draw_circle(img, cx, cy, max(1, r - max(3, r // 3)), 0.0)
    _draw_diamond(img, cx, cy, r, val)


def _draw_hexagon(img, cx, cy, r, val):
    H, W = img.shape[-2:]
    yy, xx = torch.meshgrid(torch.arange(H), torch.arange(W), indexing="ij")
    dx = (xx - cx).float().abs()
    dy = (yy - cy).float().abs()
    mask = (dx <= r) & (dy <= r * 0.866) & (dx + dy * 0.577 <= r)
    img[:, mask] = val


_DRAW_FNS = [
    _draw_circle, _draw_square, _draw_triangle, _draw_cross, _draw_diamond,
    _draw_star, _draw_ring, _draw_arrow, _draw_pentagon, _draw_hexagon,
]

print(f"Classes: {CLASS_NAMES}")

In [ ]:
class SyntheticShapeDataset(Dataset):
    def __init__(self, num_samples=5000, img_size=32, channels=3, transform=None, seed=42):
        self.num_samples = num_samples
        self.img_size = img_size
        self.channels = channels
        self.transform = transform

        rng = random.Random(seed)
        self.labels = [rng.randint(0, NUM_CLASSES - 1) for _ in range(num_samples)]
        self.params = []
        for _ in range(num_samples):
            margin = img_size // 4
            cx = rng.randint(margin, img_size - margin)
            cy = rng.randint(margin, img_size - margin)
            r = rng.randint(img_size // 6, img_size // 3)
            bg = rng.random() * 0.3
            fg = 0.5 + rng.random() * 0.5
            self.params.append((cx, cy, r, bg, fg))

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        label = self.labels[idx]
        cx, cy, r, bg, fg = self.params[idx]
        img = torch.full((self.channels, self.img_size, self.img_size), bg)
        _DRAW_FNS[label](img, cx, cy, r, fg)
        img = (img + torch.randn_like(img) * 0.05).clamp(0, 1)
        if self.transform:
            img = self.transform(img)
        return img, label


# Quick test
ds = SyntheticShapeDataset(num_samples=100, img_size=32)
img, label = ds[0]
print(f"Image shape: {img.shape}, Label: {label} ({CLASS_NAMES[label]})")
print(f"Pixel range: [{img.min():.3f}, {img.max():.3f}]")

## 2. Data Augmentation

Pure PyTorch transforms — no torchvision dependency. These reduce overfitting by creating diverse training views.

In [ ]:
class RandomHorizontalFlip:
    def __init__(self, p=0.5):
        self.p = p
    def __call__(self, img):
        return img.flip(-1) if random.random() < self.p else img


class RandomVerticalFlip:
    def __init__(self, p=0.5):
        self.p = p
    def __call__(self, img):
        return img.flip(-2) if random.random() < self.p else img


class RandomRotation90:
    def __call__(self, img):
        return torch.rot90(img, random.randint(0, 3), [-2, -1])


class ColorJitter:
    def __init__(self, brightness=0.2, contrast=0.2):
        self.brightness = brightness
        self.contrast = contrast
    def __call__(self, img):
        b = 1.0 + (random.random() * 2 - 1) * self.brightness
        c = 1.0 + (random.random() * 2 - 1) * self.contrast
        mean = img.mean()
        return ((img - mean) * c + mean) * b


class RandomErasing:
    def __init__(self, p=0.3, scale=(0.02, 0.15)):
        self.p = p
        self.scale = scale
    def __call__(self, img):
        if random.random() > self.p:
            return img
        C, H, W = img.shape
        erase_area = random.uniform(*self.scale) * H * W
        aspect = random.uniform(0.5, 2.0)
        eh = min(int(math.sqrt(erase_area * aspect)), H)
        ew = min(int(math.sqrt(erase_area / aspect)), W)
        y0 = random.randint(0, H - eh)
        x0 = random.randint(0, W - ew)
        img[:, y0:y0+eh, x0:x0+ew] = torch.rand(C, eh, ew)
        return img


class Normalize:
    def __init__(self, mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)):
        self.mean = torch.tensor(mean).view(-1, 1, 1)
        self.std = torch.tensor(std).view(-1, 1, 1)
    def __call__(self, img):
        return (img - self.mean) / self.std


class Compose:
    def __init__(self, transforms):
        self.transforms = transforms
    def __call__(self, img):
        for t in self.transforms:
            img = t(img)
        return img


# Demo augmentation
aug = Compose([RandomHorizontalFlip(), ColorJitter(), RandomErasing(p=0.5), Normalize()])
aug_img = aug(img.clone())
print(f"Original range: [{img.min():.3f}, {img.max():.3f}]")
print(f"Augmented range: [{aug_img.min():.3f}, {aug_img.max():.3f}]")

## 3. MixUp and CutMix

MixUp blends images globally; CutMix pastes rectangular patches. Both create soft labels.

In [ ]:
def mixup(images, labels, alpha=0.2):
    lam = torch.distributions.Beta(alpha, alpha).sample().item() if alpha > 0 else 1.0
    perm = torch.randperm(images.size(0))
    mixed = lam * images + (1 - lam) * images[perm]
    return mixed, labels, labels[perm], lam


def cutmix(images, labels, alpha=1.0):
    lam = torch.distributions.Beta(alpha, alpha).sample().item() if alpha > 0 else 1.0
    B, C, H, W = images.shape
    perm = torch.randperm(B)
    cut_ratio = math.sqrt(1 - lam)
    ch, cw = int(H * cut_ratio), int(W * cut_ratio)
    cy = random.randint(0, H - ch) if ch < H else 0
    cx = random.randint(0, W - cw) if cw < W else 0
    mixed = images.clone()
    mixed[:, :, cy:cy+ch, cx:cx+cw] = images[perm, :, cy:cy+ch, cx:cx+cw]
    actual_lam = 1 - (ch * cw) / (H * W)
    return mixed, labels, labels[perm], actual_lam


def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)


# Demo
batch_imgs = torch.stack([ds[i][0] for i in range(8)])
batch_labels = torch.tensor([ds[i][1] for i in range(8)])
mixed, y_a, y_b, lam = mixup(batch_imgs, batch_labels, alpha=0.4)
print(f"MixUp lambda: {lam:.3f}")
cut_mixed, y_a2, y_b2, lam2 = cutmix(batch_imgs, batch_labels)
print(f"CutMix lambda: {lam2:.3f}")

## 4. Build DataLoaders

Train set gets full augmentation; val/test only get normalization.

In [ ]:
def build_dataloaders(num_train=4000, num_val=500, num_test=500, img_size=32, batch_size=64):
    train_transform = Compose([
        RandomHorizontalFlip(p=0.5),
        RandomVerticalFlip(p=0.3),
        RandomRotation90(),
        ColorJitter(brightness=0.2, contrast=0.2),
        RandomErasing(p=0.3),
        Normalize(),
    ])
    val_transform = Compose([Normalize()])

    train_ds = SyntheticShapeDataset(num_train, img_size, transform=train_transform, seed=42)
    val_ds = SyntheticShapeDataset(num_val, img_size, transform=val_transform, seed=123)
    test_ds = SyntheticShapeDataset(num_test, img_size, transform=val_transform, seed=456)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)
    return train_loader, val_loader, test_loader


train_loader, val_loader, test_loader = build_dataloaders(
    num_train=2000, num_val=300, num_test=300, batch_size=64,
)
batch = next(iter(train_loader))
print(f"Train batches: {len(train_loader)}, batch shape: {batch[0].shape}")
print(f"Val batches:   {len(val_loader)}")
print(f"Test batches:  {len(test_loader)}")

## 5. SimpleCNN Model

A lightweight 3-layer CNN with batch normalization and dropout.

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, in_channels=3, num_classes=NUM_CLASSES):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(4),
        )
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = x.flatten(1)
        return self.classifier(x)


model = SimpleCNN().to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"SimpleCNN parameters: {total_params:,}")

# Test forward pass
dummy = torch.randn(2, 3, 32, 32, device=device)
out = model(dummy)
print(f"Input: {dummy.shape} -> Output: {out.shape}")

## 6. MiniResNet

A ResNet adapted for 32x32 images with residual (skip) connections.

In [ ]:
class BasicBlock(nn.Module):
    def __init__(self, in_planes, planes, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.shortcut = nn.Identity()
        if stride != 1 or in_planes != planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, planes, 1, stride=stride, bias=False),
                nn.BatchNorm2d(planes),
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)), inplace=True)
        out = self.bn2(self.conv2(out))
        return F.relu(out + self.shortcut(x), inplace=True)


class MiniResNet(nn.Module):
    def __init__(self, num_blocks=[2, 2, 2], num_classes=NUM_CLASSES):
        super().__init__()
        self.in_planes = 64
        self.conv1 = nn.Conv2d(3, 64, 3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.layer1 = self._make_layer(64, num_blocks[0], stride=1)
        self.layer2 = self._make_layer(128, num_blocks[1], stride=2)
        self.layer3 = self._make_layer(256, num_blocks[2], stride=2)
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(256, num_classes)
        self._init_weights()

    def _make_layer(self, planes, num_blocks, stride):
        layers = [BasicBlock(self.in_planes, planes, stride)]
        self.in_planes = planes
        for _ in range(1, num_blocks):
            layers.append(BasicBlock(planes, planes))
        return nn.Sequential(*layers)

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)), inplace=True)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.avgpool(x)
        return self.fc(x.flatten(1))


resnet = MiniResNet().to(device)
print(f"MiniResNet parameters: {sum(p.numel() for p in resnet.parameters()):,}")
out = resnet(dummy)
print(f"Input: {dummy.shape} -> Output: {out.shape}")

## 7. Transfer Learning

Freeze a pretrained backbone, attach a new head, and optionally unfreeze for fine-tuning.

In [ ]:
from contextlib import nullcontext


class TransferModel(nn.Module):
    def __init__(self, backbone, feature_dim, num_classes=NUM_CLASSES, freeze_backbone=True):
        super().__init__()
        self.backbone = backbone
        if freeze_backbone:
            for p in self.backbone.parameters():
                p.requires_grad = False
        self.head = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(feature_dim, 128),
            nn.ReLU(inplace=True),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        ctx = torch.no_grad() if not any(p.requires_grad for p in self.backbone.parameters()) else nullcontext()
        with ctx:
            features = self.backbone(x)
        return self.head(features)

    def unfreeze_backbone(self, lr_factor=0.1):
        for p in self.backbone.parameters():
            p.requires_grad = True
        return [
            {"params": self.backbone.parameters(), "lr": lr_factor},
            {"params": self.head.parameters()},
        ]


# Use MiniResNet as a "pretrained" backbone
backbone = nn.Sequential(
    resnet.conv1, resnet.bn1, nn.ReLU(inplace=True),
    resnet.layer1, resnet.layer2, resnet.layer3,
    resnet.avgpool, nn.Flatten(),
)
transfer = TransferModel(backbone, feature_dim=256).to(device)
total = sum(p.numel() for p in transfer.parameters())
trainable = sum(p.numel() for p in transfer.parameters() if p.requires_grad)
print(f"Total: {total:,}  Trainable: {trainable:,}  Frozen: {total-trainable:,}")

## 8. Label Smoothing & Cosine Warmup Scheduler

In [ ]:
class LabelSmoothingCrossEntropy(nn.Module):
    def __init__(self, smoothing=0.1):
        super().__init__()
        self.smoothing = smoothing

    def forward(self, pred, target):
        log_probs = F.log_softmax(pred, dim=-1)
        nll_loss = F.nll_loss(log_probs, target, reduction="none")
        smooth_loss = -log_probs.mean(dim=-1)
        return ((1 - self.smoothing) * nll_loss + self.smoothing * smooth_loss).mean()


class CosineWarmupScheduler(torch.optim.lr_scheduler.LRScheduler):
    def __init__(self, optimizer, warmup_epochs, total_epochs, min_lr=1e-6):
        self.warmup_epochs = warmup_epochs
        self.total_epochs = total_epochs
        self.min_lr = min_lr
        super().__init__(optimizer)

    def get_lr(self):
        if self.last_epoch < self.warmup_epochs:
            factor = self.last_epoch / max(1, self.warmup_epochs)
            return [base_lr * factor for base_lr in self.base_lrs]
        progress = (self.last_epoch - self.warmup_epochs) / max(1, self.total_epochs - self.warmup_epochs)
        cosine = 0.5 * (1 + math.cos(math.pi * progress))
        return [self.min_lr + (base_lr - self.min_lr) * cosine for base_lr in self.base_lrs]


# Demo label smoothing
criterion = LabelSmoothingCrossEntropy(smoothing=0.1)
pred = torch.randn(4, NUM_CLASSES)
target = torch.randint(0, NUM_CLASSES, (4,))
loss = criterion(pred, target)
print(f"Label smoothing loss: {loss.item():.4f}")

## 9. EMA (Exponential Moving Average)

In [ ]:
class EMA:
    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.shadow = {name: p.clone().detach() for name, p in model.named_parameters() if p.requires_grad}

    @torch.no_grad()
    def update(self, model):
        for name, p in model.named_parameters():
            if p.requires_grad and name in self.shadow:
                self.shadow[name].mul_(self.decay).add_(p.data, alpha=1 - self.decay)

    def apply(self, model):
        backup = {}
        for name, p in model.named_parameters():
            if p.requires_grad and name in self.shadow:
                backup[name] = p.data.clone()
                p.data.copy_(self.shadow[name])
        return backup

    def restore(self, model, backup):
        for name, p in model.named_parameters():
            if name in backup:
                p.data.copy_(backup[name])


print("EMA ready")

## 10. Training Loop

Full pipeline with AMP, MixUp/CutMix, EMA, cosine warmup, gradient clipping, and early stopping.

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device, use_amp=False,
                    scaler=None, use_mixup=False, ema=None, grad_clip=1.0):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    amp_dtype = torch.float16 if device.type == "cuda" else torch.bfloat16

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        use_mixed = False
        if use_mixup and torch.rand(1).item() > 0.5:
            if torch.rand(1).item() > 0.5:
                images, y_a, y_b, lam = mixup(images, labels, 0.2)
            else:
                images, y_a, y_b, lam = cutmix(images, labels, 1.0)
            use_mixed = True

        with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=use_amp):
            logits = model(images)
            loss = mixup_criterion(criterion, logits, y_a, y_b, lam) if use_mixed else criterion(logits, labels)

        optimizer.zero_grad(set_to_none=True)
        if scaler:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step()

        if ema:
            ema.update(model)
        total_loss += loss.item() * images.size(0)
        if not use_mixed:
            correct += (logits.argmax(-1) == labels).sum().item()
            total += labels.size(0)

    return {"loss": total_loss / len(loader.dataset), "accuracy": correct / max(1, total)}


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        logits = model(images)
        total_loss += criterion(logits, labels).item() * images.size(0)
        correct += (logits.argmax(-1) == labels).sum().item()
        total += labels.size(0)
    return {"loss": total_loss / total, "accuracy": correct / total}


print("Training functions defined")

In [ ]:
def train_model(model, train_loader, val_loader, epochs=15, lr=3e-3,
                use_amp=False, use_mixup=True, use_ema=True, patience=5):
    model = model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    criterion = LabelSmoothingCrossEntropy(smoothing=0.1)
    scheduler = CosineWarmupScheduler(optimizer, warmup_epochs=3, total_epochs=epochs)
    scaler = torch.amp.GradScaler() if use_amp and device.type == "cuda" else None
    ema = EMA(model, decay=0.999) if use_ema else None

    best_val_acc, best_state, no_improve = 0.0, None, 0
    history = {"train_loss": [], "val_loss": [], "val_acc": []}

    print(f"Training on {device} | epochs={epochs} | AMP={use_amp} | MixUp={use_mixup}")
    print(f"{'Ep':>3} {'TrLoss':>8} {'VLoss':>8} {'VAcc':>8} {'LR':>10}")
    print("-" * 42)

    for epoch in range(1, epochs + 1):
        tr = train_one_epoch(model, train_loader, criterion, optimizer, device,
                            use_amp=use_amp, scaler=scaler, use_mixup=use_mixup, ema=ema)
        if ema:
            backup = ema.apply(model)
            val = evaluate(model, val_loader, criterion, device)
            ema.restore(model, backup)
        else:
            val = evaluate(model, val_loader, criterion, device)
        scheduler.step()
        lr_now = optimizer.param_groups[0]["lr"]

        history["train_loss"].append(tr["loss"])
        history["val_loss"].append(val["loss"])
        history["val_acc"].append(val["accuracy"])

        print(f"{epoch:3d} {tr['loss']:8.4f} {val['loss']:8.4f} {val['accuracy']:8.4f} {lr_now:10.6f}")

        if val["accuracy"] > best_val_acc:
            best_val_acc = val["accuracy"]
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f"Early stopping at epoch {epoch}")
                break

    if best_state:
        model.load_state_dict(best_state)
    print(f"Best val accuracy: {best_val_acc:.4f}")
    return history


print("Full training pipeline ready")

## 11. Train SimpleCNN

In [ ]:
torch.manual_seed(42)
model = SimpleCNN()
history = train_model(model, train_loader, val_loader, epochs=10, lr=3e-3,
                      use_mixup=True, use_ema=True, patience=8)

## 12. Classification Metrics

Precision, recall, F1, confusion matrix, and top-k accuracy.

In [ ]:
class ClassificationMetrics:
    def __init__(self, num_classes, class_names=None):
        self.num_classes = num_classes
        self.class_names = class_names or [str(i) for i in range(num_classes)]
        self.confusion = torch.zeros(num_classes, num_classes, dtype=torch.long)
        self.all_probs = []
        self.all_labels = []

    def update(self, logits, labels):
        probs = F.softmax(logits, dim=-1)
        preds = logits.argmax(dim=-1)
        self.all_probs.append(probs.cpu())
        self.all_labels.append(labels.cpu())
        for pred, true in zip(preds.cpu(), labels.cpu()):
            self.confusion[true, pred] += 1

    def accuracy(self):
        return self.confusion.diag().sum().item() / max(1, self.confusion.sum().item())

    def per_class_metrics(self):
        metrics = {}
        for i, name in enumerate(self.class_names):
            tp = self.confusion[i, i].item()
            fp = self.confusion[:, i].sum().item() - tp
            fn = self.confusion[i, :].sum().item() - tp
            p = tp / max(1, tp + fp)
            r = tp / max(1, tp + fn)
            f1 = 2 * p * r / max(1e-8, p + r)
            metrics[name] = {"precision": p, "recall": r, "f1": f1, "support": self.confusion[i].sum().item()}
        return metrics

    def macro_f1(self):
        pc = self.per_class_metrics()
        return sum(m["f1"] for m in pc.values()) / len(pc)

    def top_k_accuracy(self, k=5):
        probs = torch.cat(self.all_probs)
        labels = torch.cat(self.all_labels)
        return (probs.topk(k, dim=-1).indices == labels.unsqueeze(1)).any(dim=1).float().mean().item()

    def print_report(self):
        print(f"\n{'Class':<12} {'Prec':>8} {'Recall':>8} {'F1':>8} {'Support':>9}")
        print("-" * 48)
        for name, m in self.per_class_metrics().items():
            print(f"{name:<12} {m['precision']:8.4f} {m['recall']:8.4f} {m['f1']:8.4f} {m['support']:9.0f}")
        print("-" * 48)
        print(f"Accuracy: {self.accuracy():.4f}  Macro F1: {self.macro_f1():.4f}  Top-3: {self.top_k_accuracy(3):.4f}")


print("Metrics class ready")

In [ ]:
@torch.no_grad()
def evaluate_full(model, loader, device):
    model.eval()
    metrics = ClassificationMetrics(NUM_CLASSES, CLASS_NAMES)
    for images, labels in loader:
        metrics.update(model(images.to(device)), labels.to(device))
    return metrics


metrics = evaluate_full(model, test_loader, device)
metrics.print_report()

## 13. Test-Time Augmentation (TTA)

Average predictions over multiple augmented views of each test image.

In [ ]:
class TTAAugmentation:
    def __init__(self, num_augmentations=5):
        self.num_augmentations = num_augmentations
        self.augmentations = [
            lambda x: x,
            lambda x: x.flip(-1),
            lambda x: x.flip(-2),
            lambda x: torch.rot90(x, 1, [-2, -1]),
            lambda x: torch.rot90(x, 2, [-2, -1]),
            lambda x: torch.rot90(x, 3, [-2, -1]),
            lambda x: x.flip(-1).flip(-2),
        ]

    @torch.no_grad()
    def predict(self, model, images):
        model.eval()
        probs = [F.softmax(model(aug(images)), dim=-1) for aug in self.augmentations[:self.num_augmentations]]
        return torch.stack(probs).mean(dim=0)


@torch.no_grad()
def evaluate_tta(model, loader, device, num_aug=5):
    model.eval()
    tta = TTAAugmentation(num_aug)
    metrics = ClassificationMetrics(NUM_CLASSES, CLASS_NAMES)
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        probs = tta.predict(model, images)
        metrics.update(torch.log(probs + 1e-8), labels)
    return metrics


tta_metrics = evaluate_tta(model, test_loader, device, num_aug=5)
print(f"Standard accuracy: {metrics.accuracy():.4f}")
print(f"TTA accuracy:      {tta_metrics.accuracy():.4f}")
print(f"Improvement:       {tta_metrics.accuracy() - metrics.accuracy():+.4f}")

## 14. Grad-CAM

Visualize which spatial regions the model attends to for each prediction.

In [ ]:
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.activations = None
        self.gradients = None
        target_layer.register_forward_hook(self._fwd_hook)
        target_layer.register_full_backward_hook(self._bwd_hook)

    def _fwd_hook(self, module, input, output):
        self.activations = output.detach()

    def _bwd_hook(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def generate(self, input_tensor, target_class=None):
        self.model.eval()
        input_tensor = input_tensor.requires_grad_(True)
        output = self.model(input_tensor)
        if target_class is None:
            target_class = output.argmax(dim=-1)
        elif isinstance(target_class, int):
            target_class = torch.tensor([target_class] * input_tensor.size(0), device=input_tensor.device)

        self.model.zero_grad()
        one_hot = torch.zeros_like(output)
        for i in range(input_tensor.size(0)):
            one_hot[i, target_class[i]] = 1.0
        output.backward(gradient=one_hot, retain_graph=True)

        weights = self.gradients.mean(dim=(-2, -1), keepdim=True)
        cam = F.relu((weights * self.activations).sum(dim=1, keepdim=True))
        cam = F.interpolate(cam, size=input_tensor.shape[-2:], mode="bilinear", align_corners=False)
        cam_min = cam.flatten(1).min(dim=1).values.view(-1, 1, 1, 1)
        cam_max = cam.flatten(1).max(dim=1).values.view(-1, 1, 1, 1)
        return ((cam - cam_min) / (cam_max - cam_min + 1e-8)).squeeze(1)


# Generate heatmaps
target_layer = model.features[-3]  # Last conv layer before pooling
grad_cam = GradCAM(model, target_layer)

test_ds = SyntheticShapeDataset(num_samples=10, img_size=32, transform=Compose([Normalize()]), seed=789)
sample_imgs = torch.stack([test_ds[i][0] for i in range(4)]).to(device)
sample_labels = [test_ds[i][1] for i in range(4)]

heatmaps = grad_cam.generate(sample_imgs)
print(f"Heatmap shape: {heatmaps.shape}")
for i in range(4):
    hot_pct = (heatmaps[i] > 0.5).float().mean()
    print(f"  Sample {i}: class={CLASS_NAMES[sample_labels[i]]:<10} max={heatmaps[i].max():.3f} hot_region={hot_pct:.1%}")

## 15. Confidence Analysis & Calibration

In [ ]:
@torch.no_grad()
def confidence_analysis(model, loader, device):
    model.eval()
    correct_confs, incorrect_confs = [], []
    all_confs, all_correct = [], []

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        probs = F.softmax(model(images), dim=-1)
        max_probs, preds = probs.max(dim=-1)
        is_correct = (preds == labels)
        correct_confs.extend(max_probs[is_correct].cpu().tolist())
        incorrect_confs.extend(max_probs[~is_correct].cpu().tolist())
        all_confs.extend(max_probs.cpu().tolist())
        all_correct.extend(is_correct.cpu().tolist())

    # ECE
    confs_t = torch.tensor(all_confs)
    correct_t = torch.tensor(all_correct, dtype=torch.float)
    bins = torch.linspace(0, 1, 11)
    ece = sum(
        (m := (confs_t >= bins[i]) & (confs_t < bins[i+1])).sum().item() / len(all_confs)
        * abs(correct_t[m].mean().item() - confs_t[m].mean().item())
        for i in range(10) if ((confs_t >= bins[i]) & (confs_t < bins[i+1])).sum() > 0
    )

    return {
        "correct_conf": sum(correct_confs) / max(1, len(correct_confs)),
        "incorrect_conf": sum(incorrect_confs) / max(1, len(incorrect_confs)),
        "ece": ece,
    }


conf = confidence_analysis(model, test_loader, device)
print(f"Mean confidence (correct):   {conf['correct_conf']:.4f}")
print(f"Mean confidence (incorrect): {conf['incorrect_conf']:.4f}")
print(f"Expected Calibration Error:  {conf['ece']:.4f}")

## 16. Inference Pipeline

In [ ]:
class ImageClassifier:
    def __init__(self, model, class_names, device, use_tta=False):
        self.model = model.to(device).eval()
        self.class_names = class_names
        self.device = device
        self.normalize = Normalize()
        self.tta = TTAAugmentation(5) if use_tta else None

    @torch.no_grad()
    def predict(self, images, top_k=3):
        images = self.normalize(images).to(self.device)
        if images.dim() == 3:
            images = images.unsqueeze(0)
        if self.tta:
            probs = self.tta.predict(self.model, images)
        else:
            probs = F.softmax(self.model(images), dim=-1)

        results = []
        for i in range(images.size(0)):
            top_probs, top_idx = probs[i].topk(top_k)
            preds = [{"class": self.class_names[j], "confidence": p.item()} for p, j in zip(top_probs, top_idx)]
            results.append({"top_class": preds[0]["class"], "confidence": preds[0]["confidence"], "predictions": preds})
        return results


classifier = ImageClassifier(model, CLASS_NAMES, device)
infer_ds = SyntheticShapeDataset(num_samples=5, img_size=32, seed=999)

for i in range(5):
    img, true_label = infer_ds[i]
    r = classifier.predict(img)[0]
    status = "correct" if r["top_class"] == CLASS_NAMES[true_label] else "wrong"
    print(f"  true={CLASS_NAMES[true_label]:<10} pred={r['top_class']:<10} conf={r['confidence']:.3f} [{status}]")

## 17. torch.compile for Inference

In [ ]:
try:
    compiled_model = torch.compile(model, mode="reduce-overhead")
    dummy = torch.randn(1, 3, 32, 32, device=device)
    with torch.no_grad():
        _ = compiled_model(dummy)  # Warmup
    print("torch.compile succeeded")
except Exception as e:
    print(f"torch.compile skipped: {e}")

## 18. Save and Load Model

In [ ]:
checkpoint = {
    "model_state_dict": model.state_dict(),
    "class_names": CLASS_NAMES,
    "num_classes": NUM_CLASSES,
    "history": history,
}
path = "/tmp/image_classifier_checkpoint.pt"
torch.save(checkpoint, path)
print(f"Saved to {path}")

loaded = torch.load(path, weights_only=True)
print(f"Loaded keys: {list(loaded.keys())}")

## 19. Key Takeaways

1. **Data augmentation is essential** — flips, jitter, and erasing reduce overfitting on small datasets
2. **MixUp/CutMix create virtual samples** — blending images smooths decision boundaries
3. **Residual connections enable depth** — skip connections let gradients flow through deep networks
4. **Transfer learning saves compute** — freeze backbone, train head, then optionally fine-tune
5. **Label smoothing improves calibration** — soft targets prevent overconfident predictions
6. **EMA stabilizes training** — moving-average weights generalize better
7. **TTA boosts accuracy for free** — average predictions over augmented test-time views
8. **Grad-CAM reveals model attention** — heatmaps show which regions drive predictions
9. **ECE measures calibration** — confidence should match accuracy

---

*Module 40 of the PyTorch Complete Learning Guide*

[← Module 39: Text Classifier](../39_text_classifier/) | [🏠 Home](../README.md)